# TCC: expansão seletiva de consultas

Este experimento é independente do notebook anterior de BM25/BGE e dos classificadores.
O código está em `src/qexp/`; mantenha o projeto completo disponível no Colab.

Etapas: `prepare` prepara corpus/índices; `generate` preenche o cache do LLM;
`run` executa A–F e retoma queries concluídas; `analyze` refaz tabelas sem executar modelos.
`run` prepara automaticamente as dependências ausentes. Use `smoke` antes de `full`.


In [ ]:
from pathlib import Path
import sys

# Local: detecta o projeto. Colab: ajuste para a pasta do repositório no Drive.
# Antes, monte o Drive em uma célula: from google.colab import drive; drive.mount('/content/drive')
PROJECT_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                     if (p / "src" / "qexp").is_dir()),
                    Path("/content/drive/MyDrive/PLN"))
if not (PROJECT_ROOT / "src" / "qexp").is_dir():
    raise FileNotFoundError("Ajuste PROJECT_ROOT para a pasta do projeto, contendo src/ e configs/.")
PROJECT_ROOT = PROJECT_ROOT.resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))
print("Projeto:", PROJECT_ROOT)


In [ ]:
# Execute uma vez no ambiente/Colab. O pacote precisa estar disponível junto ao notebook.
%pip install -e "{PROJECT_ROOT}[notebooks]"


In [ ]:
from qexp.cli import load_settings, execute_stage

PROFILE = "smoke"  # "full" para todas as queries dos três datasets
STAGE = "run"      # prepare, generate, run ou analyze
RESUME = True
settings = load_settings(PROJECT_ROOT / "configs" / f"{PROFILE}.json")
settings


## Execução
O progresso é salvo por query. Reexecute com a mesma configuração para retomar.
Mudanças de código, parâmetros ou dados invalidam checkpoints correspondentes.
Use uma nova pasta de cache de geração e `RESUME=False` para novas medições de custo.
Fixe as revisões de LLM/BGE antes da execução final.


In [ ]:
result = execute_stage(STAGE, settings, resume=RESUME)
if "summary" in result:
    display(result["summary"].round(4))
    display(result["hypotheses"].round(4))
    display(result["degradation"].round(4))
else:
    print(result)


## Visualização dos resultados salvos
Esta célula pode ser executada depois da configuração, sem executar modelos novamente.
Os tempos são custos compostos; checkpoints preservam os tempos da execução de origem.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

output = Path(settings["output_dir"])
summary = pd.read_csv(output / "qexp_summary.csv")
metadata = __import__("json").loads((output / "qexp_metadata.json").read_text())
k, n = metadata["config"]["eval_k"], metadata["config"]["candidate_n"]
figure, axes = plt.subplots(1, 3, figsize=(19, 5))
for axis, metric, title in zip(axes,
        [f"ndcg@{k}", f"candidate_recall@{n}", "avg_estimated_uncached_sec"],
        ["Qualidade final", "Recall dos candidatos", "Custo composto estimado (s/query)"]):
    sns.barplot(data=summary, x="pipeline_id", y=metric, hue="dataset", ax=axis)
    axis.set_title(title)
figure.tight_layout()
figure.savefig(output / "qexp_quality_cost.png", dpi=160, bbox_inches="tight")
plt.show()


In [ ]:
# ZIP apenas deste experimento. No Colab, use files.download(archive).
import shutil
archive = shutil.make_archive(str(output), "zip", output)
print(archive)
